In [ ]:
import scanpy as sc
import os

folder = "./data/topovelo-output/"  # change if different

files = [f for f in os.listdir(folder) if f.endswith(".h5ad")]
files

In [ ]:
adata = sc.read_h5ad("./data/topovelo-output/curio-seq-embryoid-out.h5ad")
adata

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc
import numpy as np

# If you're not sure the key exists:
assert "X_spatial" in adata.obsm, "No spatial coordinates found!"

coords = adata.obsm["X_spatial"]

plt.figure(figsize=(6, 6))
plt.scatter(
    coords[:, 0],
    coords[:, 1],
    s=20,
    c=adata.obs["clusters"].astype("category").cat.codes,
    cmap="tab20",
    alpha=0.8,
    linewidths=0
)
plt.gca().set_aspect("equal", "box")
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
from scipy.sparse import issparse
import numpy as np
from scripts.VectorFieldEmbedder import *
import matplotlib.pyplot as plt

# -----------------
# 1. Expression
# -----------------
X = adata.layers["spliced"]
if issparse(X):
    X = X.toarray()

# -----------------
# 2. Velocity
# -----------------
V = adata.layers["gat_velocity_u"]
if issparse(V):
    V = V.toarray()

# -----------------
# 3. Embedding (spatial)
# -----------------
X_emb = adata.obsm["X_spatial"].astype(float)

# -----------------
# 4. Cluster colors
# -----------------
scatter_color = adata.obs["clusters"].astype("category").cat.codes

# -----------------
# 5. Build TPS embedding
# -----------------
emb = VectorFieldEmbedder(
    X,
    V,
    dist_method="phase",
    dof=50,
    X_emb=X_emb,
    alpha=0.75,
    knn_k=30,
    max_tps_points=4000
)

emb.initialize_embedding(seed=42)

In [ ]:
from scripts.plotting import *

plot_velocity_streamplot(
    X_2d=emb.X_emb,
    tps_vf=emb.tps_vf,
    scatter_color=scatter_color,
    grid_density=1,
    stream_density=1.2,
    scatter_size=200,
    scatter_alpha=0.3,
    figsize=(6, 6),
    aspect=1,
    vmin=0.0,
    vmax=1.0,
    cmap="tab10",
    grid_size=30,
    show_labels=False
)